# 02 — GOSS and EFB: how LightGBM gets light

*Chapter 10 · LightGBM · Notebook 2 of 5*

Notebook 1 changed *how the trees grow* (leaf-wise). This notebook changes *what each tree is trained
on*, so that growing them on large data stays cheap. LightGBM leans on two ideas for this:

- **GOSS** — Gradient-based One-Side Sampling — trains each tree on **fewer rows**, chosen by how much
  they still matter, instead of all of them.
- **EFB** — Exclusive Feature Bundling — packs **fewer features** by bundling sparse columns that rarely
  fire together.

We **build GOSS by hand** and measure exactly what it buys (and, honestly, when it buys nothing), then
**name** EFB and see its idea on a small example. The pattern is Chapter 09 NB 4's: build one, name one.

**Prerequisites.** Chapter 08 (the per-row gradient `g`); Chapter 09 NB 2 (the split-gain is built from
gradient sums); Chapter 00 (train/test, accuracy/AUC).

**What you'll be able to do.**
- Explain GOSS in one sentence and build its `(1−a)/b` reweight from scratch, keeping the gradient
  sums unbiased.
- Measure *when* GOSS gives a more precise split than a uniform subsample — and when it does not.
- State GOSS's honest payoff regime, and what EFB does, without overselling either.

## A row's influence is its gradient

Two facts from earlier chapters set this up.

**The split-gain is a sum over rows.** Chapter 09 NB 2 scored a split by the *structure-score gain*

`gain = ½ [ G_L²/(H_L+λ) + G_R²/(H_R+λ) − G²/(H+λ) ]`,

where `G = Σ gᵢ` and `H = Σ hᵢ` add up the per-row gradient `g` and curvature `h` over the rows in a
node. Every quantity a tree needs to choose a split is an **accumulation over rows**.

**The gradient is how wrong a row still is.** For squared error with a constant first prediction
`F₀ = mean(y)`, the gradient is `g = F₀ − y` (Chapter 08): the leftover residual. A row with large `|g|`
is poorly fit and pulls hard on the next tree; a row with tiny `|g|` is already well predicted and barely
moves the sums. So `|g|` is a row's **influence** on the next tree.

That suggests a question: if influence is so uneven, must every tree look at every row? GOSS says no.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()
SEED = 0
rng = np.random.default_rng(SEED)

# A regression toy, so the gradient is the plain residual (ch 08). With a constant
# first prediction F0 = mean(y), the gradient of 1/2 (y - F)^2 at F0 is g = F0 - y,
# and the curvature h = 1. The size |g| = |F0 - y| is a row's INFLUENCE on the next tree.
n = 2000
x = rng.normal(size=n)
y = 2.0 * x + rng.normal(scale=1.0, size=n)
F0 = float(y.mean())
g = F0 - y                 # squared-error gradient at F0  (= -(residual))
h = np.ones(n)             # squared-error curvature
absg = np.abs(g)

print(f"n = {n}   F0 = mean(y) = {F0:.4f}")
print(f"|g| spread:  min {absg.min():.3f}   median {np.median(absg):.3f}   max {absg.max():.3f}")


## Two ways to use fewer rows

**The honest baseline: uniform subsampling.** Keep a fraction `f` of the rows at random, drop the rest.
To keep the gradient sums on the same scale, re-weight each kept row by `1/f` (so `f·n` rows standing in
for `n` still add up to the full total in expectation). This is exactly *stochastic gradient boosting*
(Friedman 2002). Its blind spot: chance alone decides who stays, so it discards a large-`|g|` row — one
that matters a lot — as readily as a tiny one.

**GOSS: sample one side only.** Split the rows by influence:

1. **Keep** the top `a` fraction by `|g|` — the most influential rows — *exactly*, no sampling.
2. **Sample** a fraction `b` of the remaining small-`|g|` rows at random.
3. **Up-weight** those sampled rows by `(1−a)/b` so the small-gradient side still counts for its full
   share.

"One-side" sampling: we only gamble on the rows that barely matter. The influential rows are always in.
We use `a = 0.2`, `b = 0.1` — keeping `a + b = 30%` of the rows.

In [ ]:
a, b = 0.20, 0.10                 # top_rate, other_rate (LightGBM's names)
frac = a + b                         # fraction of rows actually used
n_top, n_other = int(a * n), int(b * n)

ranked = np.argsort(-absg)           # row indices ordered by |gradient|, largest first
top_idx = ranked[:n_top]             # the large-|g| rows: KEPT exactly
rest_idx = ranked[n_top:]            # the small-|g| pool: we SAMPLE from this side


def goss_weights(seed):
    """GOSS row weights: 1.0 on kept top rows, (1-a)/b on a b-sample of the rest, else 0."""
    r = np.random.default_rng(seed)
    sampled = r.choice(rest_idx, size=n_other, replace=False)
    w = np.zeros(n)
    w[top_idx] = 1.0
    w[sampled] = (1.0 - a) / b
    return w


print(f"keep top a = {a:.0%}  ({n_top} rows, kept exactly)")
print(f"sample  b = {b:.0%}  ({n_other} of the {len(rest_idx)} small-|g| rows)")
print(f"-> rows used per tree = a + b = {frac:.0%}")
print(f"-> sampled rows up-weighted by (1 - a)/b = {(1.0 - a) / b:.1f}")


In [ ]:
cutoff = absg[ranked[n_top - 1]]     # the |g| of the smallest row we still keep

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.hist(absg, bins=40, color=COLORS["muted"], edgecolor=COLORS["text"], linewidth=0.3)
ax.axvspan(cutoff, absg.max(), color=COLORS["highlight"], alpha=0.35,
           label=f"kept exactly (top {a:.0%}, |g| ≥ {cutoff:.2f})")
ax.axvline(cutoff, color=COLORS["model"], linewidth=2)
ax.set_xlabel("|gradient|  (a row's influence on the next tree)")
ax.set_ylabel("number of rows")
ax.set_title("GOSS keeps the large-|g| rows, samples the crowd")
ax.legend()
plt.show()


**Read the figure.** The bars are the distribution of row influence `|g|`. Most rows sit on the
left — small residuals, little pull on the next tree. The shaded band on the right is the top `20%` by
`|g|`: GOSS keeps **every** row there, no questions asked. From the unshaded majority it keeps only a
`10%` sample. The bet is that the left side is safe to thin out — and the reweight is what makes thinning
it unbiased.

## Why the weight is `(1−a)/b`

We thinned the small-`|g|` side, so we have to scale it back up. The small side holds `(1−a)·n` rows;
we kept a sample of `b·n` of them. Each sampled row therefore stands in for

`(1−a)·n / (b·n) = (1−a)/b`

of its peers — here `(1−0.2)/0.1 = 8`. Give every sampled row that weight and the **expected** gradient
sum is restored exactly:

`E[ Σ_sampled (1−a)/b · g ] = (1−a)/b · (b·n / ((1−a)·n)) · Σ_small g = Σ_small g`.

The kept top rows carry weight `1`, so `E[G_GOSS] = Σ_top g + Σ_small g = G_full` — **unbiased**. And the
Hessian sum is recovered not only in expectation but *exactly*, because the counts are fixed:

`H_GOSS = (1)·a·n + ((1−a)/b)·(b·n) = a·n + (1−a)·n = n = H_full`.

Let's confirm both on the toy.

In [ ]:
N_MC = 400
G_full, H_full = float(g.sum()), float(h.sum())
m_unif = int(round(frac * n))

goss_G, goss_H, unif_G = [], [], []
for s in range(N_MC):
    w = goss_weights(2000 + s)
    goss_G.append(float((w * g).sum()))
    goss_H.append(float((w * h).sum()))
    u = np.random.default_rng(2000 + s).choice(n, size=m_unif, replace=False)
    unif_G.append(float(g[u].sum() / frac))     # uniform with the 1/f reweight

print(f"full data:       G = {G_full:+8.3f}      H = {H_full:.1f}")
print(f"GOSS estimate:   G mean {np.mean(goss_G):+8.3f} (std {np.std(goss_G):5.1f})   "
      f"H mean {np.mean(goss_H):.3f} (std {np.std(goss_H):.3f})")
print(f"uniform estimate:G mean {np.mean(unif_G):+8.3f} (std {np.std(unif_G):5.1f})")
print(f"concentration here (top 20% share of sum|g|) = "
      f"{np.sort(absg)[::-1][:int(0.2 * n)].sum() / absg.sum():.2f}")


## Unbiased is not the whole story — variance is

Both estimators are unbiased (`H` is even exact for GOSS), so on average they agree with the full data.
What separates them is **scatter**: how far a single subsample's gain estimate strays from the truth. The
gain is built from `G²/(H+λ)`, and the largest-`|g|` rows dominate those sums. GOSS keeps those rows
*exactly* — they contribute **zero** sampling variance — and only gambles on the small rows, which move
the sums little. Uniform sampling gambles on **all** rows, including the influential ones.

So GOSS should give a *tighter* gain estimate — **but only when a few rows really do dominate**. Notice
the concentration printed above is about `0.44`: on this round-1 toy the gradients are roughly Gaussian,
so no single row dominates strongly (and indeed GOSS's spread above is no better than uniform's here). Let
us look first at a set where a few rows clearly do dominate.

In [ ]:
def weighted_gain(left, w, gg, hh, lam=0.0):
    """Half the structure-score gain (ch 09 NB 2) for a boolean left-mask, with row weights w."""
    wg, wh = w * gg, w * hh
    GL, HL = wg[left].sum(), wh[left].sum()
    GR, HR = wg[~left].sum(), wh[~left].sum()
    G, H = wg.sum(), wh.sum()
    if HL <= 1e-9 or HR <= 1e-9:
        return -np.inf
    return 0.5 * (GL**2 / (HL + lam) + GR**2 / (HR + lam) - G**2 / (H + lam))


def concentration(grad):
    """Share of the total sum|g| held by the top 20% of rows -- 0.2 = flat, 1.0 = one row."""
    ag = np.abs(grad)
    return float(np.sort(ag)[::-1][:int(0.2 * len(ag))].sum() / ag.sum())


# A CONCENTRATED gradient set: a few rows carry most of sum|g| (a heavy tail). This is
# the regime GOSS was built for -- later boosting rounds, where most rows fit well.
rc = np.random.default_rng(7)
g_c = rc.choice([-1.0, 1.0], size=n) * rc.uniform(0, 1, n) ** 4      # tail power 4 -> concentrated
x_c = g_c + rc.normal(scale=0.15, size=n)                           # a feature that tracks |g|
h_c = np.ones(n)
conc_c = concentration(g_c)

ranked_c = np.argsort(-np.abs(g_c))
top_c, rest_c = ranked_c[:n_top], ranked_c[n_top:]
grid = np.quantile(x_c, np.linspace(0.05, 0.95, 50))
gains_full = [weighted_gain(x_c <= t, np.ones(n), g_c, h_c) for t in grid]
t_star, gain_true = grid[int(np.argmax(gains_full))], max(gains_full)

goss_gain, unif_gain = [], []
for s in range(400):
    r = np.random.default_rng(900 + s)
    samp = r.choice(rest_c, size=n_other, replace=False)
    wg = np.zeros(n)
    wg[top_c] = 1.0
    wg[samp] = (1.0 - a) / b
    u = r.choice(n, size=m_unif, replace=False)
    wu = np.zeros(n)
    wu[u] = 1.0 / frac
    goss_gain.append(weighted_gain(x_c <= t_star, wg, g_c, h_c))
    unif_gain.append(weighted_gain(x_c <= t_star, wu, g_c, h_c))
goss_gain, unif_gain = np.array(goss_gain), np.array(unif_gain)

print(f"gradient concentration = {conc_c:.2f}   true gain = {gain_true:.2f}")
print(f"GOSS    gain estimate: mean {goss_gain.mean():.2f}  std {goss_gain.std():.2f}")
print(f"uniform gain estimate: mean {unif_gain.mean():.2f}  std {unif_gain.std():.2f}")
print(f"std ratio GOSS/uniform = {goss_gain.std() / unif_gain.std():.2f}")

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.hist(unif_gain, bins=30, color=COLORS["test"], alpha=0.65, label="uniform subsample")
ax.hist(goss_gain, bins=30, color=COLORS["model"], alpha=0.65, label="GOSS")
ax.axvline(gain_true, color=COLORS["text"], linestyle="--", linewidth=1.5, label="full-data gain")
ax.set_xlabel("estimated split gain (each from the same 30% of rows)")
ax.set_ylabel("count over 400 subsample draws")
ax.set_title(f"concentrated gradients (concentration {conc_c:.2f}): GOSS is tighter")
ax.legend()
plt.show()


**Read the figure.** Both histograms sit on top of the full-data gain (dashed line) — both are
unbiased. But the uniform estimates (amber) spread wide, while the GOSS estimates (blue) cluster tightly:
about **three times** smaller scatter for the same `30%` of rows. The reason is the one above — GOSS holds
the dominant rows fixed and samples only the rest. With concentrated gradients, that is most of the
variance gone.

## When does GOSS actually win? It depends on concentration

The advantage hung on a condition: *a few rows dominate*. Let us make that precise. Define a node's
**gradient concentration** as the share of `Σ|g|` carried by its top `20%` of rows — `0.2` means perfectly
flat (every row equal), values near `1` mean a handful of rows hold almost all the influence.

The claim is that GOSS beats a uniform subsample **above** a concentration threshold, and **loses below
it**: when no row dominates, holding the top `20%` fixed buys nothing, while the `×8` up-weight on the
sampled rows actually *adds* variance. (Ke et al.'s Theorem 3.2 points the same way — it bounds GOSS's
gain error by a term that grows with the largest *sampled* gradient, which is small exactly when the
gradients are concentrated. The head-to-head crossover against uniform sampling below is our own
measurement, consistent with that bound.) We sweep concentration from flat to heavy-tailed and watch the
crossover.

In [ ]:
def gain_std_ratio(grad, feat, n_mc=300, base=1500):
    """std(GOSS gain estimate) / std(uniform gain estimate) at the full-data best split."""
    order = np.argsort(-np.abs(grad))
    top, rest = order[:n_top], order[n_top:]
    hh = np.ones(len(grad))
    gr = np.quantile(feat, np.linspace(0.05, 0.95, 50))
    t = gr[int(np.argmax([weighted_gain(feat <= tt, np.ones(len(grad)), grad, hh) for tt in gr]))]
    gs, us = [], []
    for s in range(n_mc):
        r = np.random.default_rng(base + s)
        samp = r.choice(rest, size=n_other, replace=False)
        wg = np.zeros(len(grad))
        wg[top] = 1.0
        wg[samp] = (1.0 - a) / b
        u = r.choice(len(grad), size=m_unif, replace=False)
        wu = np.zeros(len(grad))
        wu[u] = 1.0 / frac
        gs.append(weighted_gain(feat <= t, wg, grad, hh))
        us.append(weighted_gain(feat <= t, wu, grad, hh))
    return np.std(gs) / np.std(us)


concs, ratios = [], []
for k in [0.0, 1.0, 2.0, 3.0, 4.0, 6.0, 8.0]:
    rk = np.random.default_rng(7)
    gk = rk.choice([-1.0, 1.0], size=n) * rk.uniform(0, 1, n) ** k
    xk = gk + rk.normal(scale=0.15, size=n)
    concs.append(concentration(gk))
    ratios.append(gain_std_ratio(gk, xk))
    print(f"  concentration {concs[-1]:.2f}:  GOSS/uniform std ratio = {ratios[-1]:.2f}")

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.axhspan(1.0, max(ratios) * 1.05, color=COLORS["test"], alpha=0.12)
ax.axhspan(min(ratios) * 0.9, 1.0, color=COLORS["model"], alpha=0.12)
ax.plot(concs, ratios, "o-", color=COLORS["text"], linewidth=2.2, zorder=3)
ax.axhline(1.0, color=COLORS["error"], linestyle="--", linewidth=1.2)
ax.axvspan(0.21, 0.47, color=COLORS["highlight"], alpha=0.30,
           label="dense tabular gradients (measured, rounds 1-200)")
ax.text(0.62, max(ratios) * 0.92, "uniform tighter", color=COLORS["test"])
ax.text(0.62, min(ratios) * 1.1, "GOSS tighter", color=COLORS["model"])
ax.set_xlabel("gradient concentration (top-20% share of Σ|g|)")
ax.set_ylabel("std ratio  GOSS / uniform")
ax.set_title("GOSS wins only when gradients are concentrated")
ax.legend(loc="upper right")
plt.show()


**Read the figure.** Below a concentration of about `0.5` the ratio is **above 1** — uniform
sampling gives the tighter gain estimate, because GOSS's `×8` reweight inflates the variance of rows that
were not special. Above `0.5` the curve drops fast: at concentration `0.66` GOSS's scatter is about a
third of uniform's, and lower still by `0.86`. The shaded band is where **dense tabular gradients actually
live** — measured on real boosting runs, concentration climbs only from about `0.21` (first round) to
`0.47` (after 200 rounds). That band sits **at or slightly below** the crossover: on dense tabular data
GOSS is at best **tied** with uniform sampling, often a touch behind. Its decisive advantage needs the
heavily concentrated gradients of very large, wide, or sparse problems. (The crossover's exact location
depends a little on the split feature — it sits near `0.5` for a generic feature and shifts lower, GOSS
winning sooner, when the split itself separates high- from low-gradient rows; the *direction* of the
effect is robust.)

## What this means in practice

Two honest consequences, both measured below:

- **Quality.** On a moderate dense dataset, training on GOSS's `30%` of the rows reaches essentially the
  **same** held-out accuracy as training on all of them — GOSS keeps the signal. But a plain uniform
  `30%` subsample does as well here, for the reason the concentration-crossover figure gives.
- **Speed.** GOSS does fewer row-passes, but on dense moderate data the wall-clock is **flat** — the
  histogram build over features dominates, not the row count. GOSS's time saving shows up when the data
  is large enough that the row-pass dominates, the same regime where its gradients are concentrated.

So GOSS is not a free accuracy or speed win on every dataset. It is the tool for the **large / wide /
sparse** regime — which is exactly where Notebook 5 will dial the data size up to find it. Let's switch it
on in LightGBM and confirm both points.

In [ ]:
import time

from lightgbm import LGBMClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

Xc, yc = make_classification(n_samples=60000, n_features=30, n_informative=12,
                             n_redundant=6, flip_y=0.06, class_sep=0.8, random_state=SEED)
Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.25, random_state=SEED)


def fit_report(label, **kw):
    m = LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=SEED, **kw)
    t0 = time.perf_counter()
    m.fit(Xtr, ytr)                          # LightGBM's info banner stays visible
    dt = time.perf_counter() - t0
    p = m.predict_proba(Xte)[:, 1]
    print(f"  {label:34s} acc {accuracy_score(yte, p >= 0.5):.4f}   "
          f"auc {roc_auc_score(yte, p):.4f}   fit {dt:5.2f}s")


print(f"held-out quality on {Xtr.shape[0]} train / {Xte.shape[0]} test rows, 300 trees:")
fit_report("full data (all rows)")
fit_report("GOSS (top 0.2, other 0.1 -> 30%)", data_sample_strategy="goss",
           top_rate=0.2, other_rate=0.1)
fit_report("uniform subsample (30%)", subsample=0.3, subsample_freq=1)


**Read the numbers.** All three land within a few thousandths of each other on accuracy and AUC:
GOSS on `30%` of the rows matches the full-data model, and the uniform `30%` subsample matches it too —
as the concentration-crossover figure leads us to expect for dense data at or below it. The fit times are
nearly identical here:
on `45 000 × 30` the per-tree cost is dominated by building feature histograms, not by reading rows, so
using fewer rows does not move the clock. (On a much larger row count the row-pass starts to dominate and
GOSS's timing pulls ahead — that is Notebook 5's territory.) GOSS earns its place not as a universal win
but as the right tool when `n` is large and the gradients are concentrated.

## EFB — bundling features that never fire together (named)

GOSS thins the **rows**; **Exclusive Feature Bundling** thins the **columns**. The cost of building a
tree's histograms scales with the number of features, and on sparse data many features are
**approximately exclusive** — they are rarely nonzero at the same time. The clearest case is one-hot
encoding: the columns from a single categorical are *mutually* exclusive (exactly one is `1` per row).
Bag-of-words columns for rare terms are nearly so.

EFB bundles such features into one — laying their value ranges end to end (offsets), so a single bundled
feature carries all of them. It tolerates a small **conflict rate** (rows where two bundled features are
both nonzero), which is what makes it more than concatenating perfectly-exclusive one-hot columns: a few
collisions are accepted in exchange for far fewer features. The histogram cost drops from
`O(n · #features)` to `O(n · #bundles)`. We do not build EFB here; we look at the exclusivity it exploits.

In [ ]:
# Three categoricals (6 / 5 / 4 levels) one-hot encoded -> 15 columns in 3 exclusive groups.
rng_efb = np.random.default_rng(SEED)
N = 5000
levels = [6, 5, 4]
blocks = [np.eye(L)[rng_efb.integers(0, L, size=N)] for L in levels]
OH = np.hstack(blocks)
n_cols, n_bundles = OH.shape[1], len(levels)

same_group = float((OH[:, 0] * OH[:, 1]).mean())          # cols 0,1 -> same categorical
diff_group = float((OH[:, 0] * OH[:, levels[0]]).mean())  # col 0 vs another group's col
print(f"one-hot columns: {n_cols}  ->  EFB bundles to ~{n_bundles} (one per exclusive group)")
print(f"conflict rate, same-group columns:      {same_group:.4f}  (mutually exclusive -> 0)")
print(f"conflict rate, different-group columns: {diff_group:.4f}  (EFB tolerates a small rate)")

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.imshow(OH[:60].T, aspect="auto", cmap=viz.CMAP_PROBA, interpolation="nearest")
for boundary in np.cumsum(levels)[:-1]:
    ax.axhline(boundary - 0.5, color=COLORS["text"], linewidth=1.5)
ax.set_xlabel("rows (first 60)")
ax.set_ylabel("one-hot columns")
ax.set_title("three exclusive groups (lines) -> three bundles")
plt.show()


**Read the figure.** Each column of the image is one row; each horizontal strip is a one-hot
feature, lit where it is `1`. Within each of the three groups (separated by the lines) exactly **one**
strip fires per row — the same-group conflict rate is `0`. Across groups, collisions are common, so those
features cannot bundle. EFB packs the `15` columns into `3` bundles, one per exclusive group, and on truly
sparse data (text, high-cardinality categoricals) the reduction is far larger — that is where LightGBM's
histogram build gets cheap.

## Your turn

1. **(easy)** In the GOSS build, change `a` and `b` (keep `a + b = 0.3`, e.g. `a=0.05, b=0.25` vs
   `a=0.25, b=0.05`). Which split keeps more of the influential rows, and how does the reweight `(1−a)/b`
   change?
2. **(core)** Re-run the concentration-sweep figure with a different feature noise scale in `x_c`. Does
   the crossover concentration move? (It should: the crossover sits near `0.5` for a generic feature and
   lower when the split separates high- from low-gradient rows.) Explain why holding the top rows fixed
   stops helping when no row dominates.
3. **(reach)** Take the `make_classification` data, fit a `GradientBoostingClassifier` for `1`, `20`, and
   `100` rounds, compute the gradient `g = p − y` each time, and measure the concentration. Does it rise
   with boosting rounds? Relate that to *when in training* GOSS helps most.

## What you built

You built **GOSS** by hand: rank rows by influence `|g|`, keep the top `a` exactly, sample `b` of the
rest, and up-weight the sample by `(1−a)/b`. You proved the reweight keeps the gradient sums unbiased
(`H` exactly), then measured the part that is easy to oversell: GOSS gives a **lower-variance** split
estimate than uniform sampling **only when the gradients are concentrated** (crossover near `0.5`).
On dense tabular data, where concentration stays modest, GOSS matches full-data quality on a fraction of
the rows but merely ties uniform sampling, at flat wall-clock — its real payoff is the large / wide /
sparse regime. You then **named EFB**, the matching trick for sparse *features*.

**Vocabulary.** GOSS · top_rate `a` / other_rate `b` · the `(1−a)/b` reweight · gradient concentration ·
EFB · conflict rate · `data_sample_strategy='goss'`.

Next: **the optimal categorical split** — how LightGBM splits a categorical feature without trying all
`2^(K−1)−1` partitions.

## References

- Ke, G., Meng, Q., Finley, T., et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision
  Tree.* Advances in Neural Information Processing Systems 30 (NeurIPS 2017). (GOSS — §3.2 and Theorem
  3.2; EFB — §4.)
- Friedman, J. H. (2002). *Stochastic Gradient Boosting.* Computational Statistics & Data Analysis 38(4),
  367-378. DOI [10.1016/S0167-9473(01)00065-2](https://doi.org/10.1016/S0167-9473(01)00065-2). (The
  uniform-subsample baseline.)
- Chen, T., & Guestrin, C. (2016). *XGBoost: A Scalable Tree Boosting System.* KDD '16. DOI
  [10.1145/2939672.2939785](https://doi.org/10.1145/2939672.2939785). (The structure-score gain — Chapter
  09 NB 2.)

*Previous: 01 — Leaf-wise growth.*  ·  *Next: 03 — The optimal categorical split.*